# LeetCode 2235 — Add Two Integers bằng Python

Link đề: https://leetcode.com/problems/add-two-integers/ · Độ khó: **Easy** · Chủ đề: Math

Skeleton LeetCode:

```python
class Solution:
    def sum(self, num1: int, num2: int) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Làm quen với cấu trúc chung của một bài LeetCode: `class Solution`, tên phương thức, kiểu tham số/giá trị trả về.
2. Hiểu số nguyên trong Python (`int`): không tràn số, có thể âm, và cách `+` hoạt động.
3. Viết 4 cách: cộng trực tiếp `+`, cộng bằng vòng lặp tăng/giảm từng đơn vị, cộng bằng **thao tác bit** (XOR + carry), và dùng `sum(...)`/`operator.add`.
4. Hiểu bộ cộng nửa (half adder): `XOR` là tổng không nhớ, `AND << 1` là phần nhớ; và cách xử lý **số âm** bằng mặt nạ 32 bit.
5. Nhận biết vì sao tên phương thức `sum` không "đè" hàm `sum` có sẵn, và tránh các lỗi người mới hay gặp.

## 2. Đề bài & ví dụ

Cho hai số nguyên `num1` và `num2`. Hãy trả về **tổng** của chúng.

| Ví dụ | num1 | num2 | Output |
|---|---|---|---|
| 1 | `12` | `5` | `17` |
| 2 | `-10` | `4` | `-6` |

Constraints: `-100 <= num1, num2 <= 100`.

Đây là bài nhập môn dùng để làm quen với giao diện LeetCode. Đề rất đơn giản, nhưng ta sẽ tận dụng để học nền tảng về số nguyên và phép cộng nhị phân.

In [1]:
examples = [(12, 5, 17), (-10, 4, -6)]
for a, b, expected in examples:
    print(f"num1={a:>4}, num2={b:>3} | mong đợi = {expected} | a + b = {a + b}")

num1=  12, num2=  5 | mong đợi = 17 | a + b = 17
num1= -10, num2=  4 | mong đợi = -6 | a + b = -6


## 3. Kiến thức nền

### 3.1. Cấu trúc một bài LeetCode

LeetCode cho sẵn một **lớp** (`class Solution`) và một **phương thức** (hàm nằm trong lớp). Tham số đầu tiên luôn là `self` (đối tượng hiện tại, ta không cần dùng tới). Chú thích kiểu `num1: int` và `-> int` chỉ là gợi ý cho người đọc; Python không ép kiểu.

Để thử, ta tạo một đối tượng `Solution()` rồi gọi phương thức.

In [2]:
class Demo:
    def hello(self, name: str) -> str:
        return "Xin chào " + name

d = Demo()
print(d.hello("LeetCode"))

Xin chào LeetCode


### 3.2. Số nguyên `int` của Python

Python dùng số nguyên **độ dài tùy ý**: không bị tràn số như `int` 32 bit của C++/Java. Số âm là số bình thường. Phép `+`, `-`, `*`, `//` (chia nguyên), `%` (chia lấy dư) hoạt động đúng với cả số âm.

In [3]:
print(12 + 5, -10 + 4, -3 + -4)
print(2**100 + 1)           # số rất lớn vẫn đúng
print(type(5), type(-5))

17 -6 -7
1267650600228229401496703205377
<class 'int'> <class 'int'>


### 3.3. Tên `sum` — phương thức và hàm có sẵn

Python có sẵn hàm `sum(iterable)` (cộng mọi phần tử). Đề này bắt ta đặt tên phương thức là `sum`. Điều đó **không** gây xung đột: phương thức nằm trong **lớp**, còn trong thân phương thức, tên `sum` đơn thuần vẫn trỏ tới hàm có sẵn (vì phạm vi lớp không "bọc" thân hàm). Muốn gọi phương thức, ta phải viết `self.sum(...)` hoặc `Solution().sum(...)`.

In [4]:
class T:
    def sum(self, a, b):
        return sum([a, b])        # sum ở đây là hàm có sẵn của Python

print(T().sum(3, 4))
print(sum([1, 2, 3]))

7
6


### 3.4. Số nhị phân và cộng nhị phân bằng tay

`5 = 101`, `3 = 011`. Cộng từng cột từ phải sang trái, nhớ `1` nếu tổng cột `>= 2`:

```
  1 0 1
+ 0 1 1
-------
1 0 0 0   (= 8)
```

Cột 0: `1+1 = 10` → viết `0` nhớ `1`. Cột 1: `0+1+1(nhớ) = 10` → viết `0` nhớ `1`. Cột 2: `1+0+1 = 10` → viết `0` nhớ `1`. Cột 3: viết `1`.

In [5]:
print(f"{5:04b} + {3:04b} = {5 + 3:04b}")
print(bin(5), bin(3), bin(8))

0101 + 0011 = 1000
0b101 0b11 0b1000


### 3.5. Bộ cộng nửa: XOR và AND

Với **hai bit** `a`, `b`: tổng (không nhớ) là `a ^ b`, phần nhớ là `a & b` (nhớ sang cột kế tiếp nên dịch trái `<< 1`). Với **cả số**, `a ^ b` là "tổng từng cột mà bỏ qua nhớ" và `(a & b) << 1` là "tất cả các nhớ". Cộng hai kết quả này (lặp lại tới khi không còn nhớ) ra tổng thật.

In [6]:
a, b = 5, 3
s = a ^ b                 # tổng không nhớ
c = (a & b) << 1          # nhớ
print(f"{a:04b} ^ {b:04b} = {s:04b} | ({a:04b} & {b:04b}) << 1 = {c:04b}")
print("s + c =", s + c, "== a + b =", a + b)

0101 ^ 0011 = 0110 | (0101 & 0011) << 1 = 0010
s + c = 8 == a + b = 8


### 3.6. Số âm và mặt nạ 32 bit

Trong máy tính, số âm lưu theo **bù hai**. Python thì coi số âm như có vô hạn bit `1` ở đầu, nên vòng lặp cộng bit với số âm có thể **không bao giờ dừng**. Cách xử lý: dùng mặt nạ `mask = 0xFFFFFFFF` (32 bit `1`) để "cắt" mọi thứ về 32 bit, rồi nếu kết quả `> 0x7FFFFFFF` (bit dấu bằng `1`) thì đổi lại thành số âm bằng `~(x ^ mask)`.

In [7]:
mask = 0xFFFFFFFF
print(-1 & mask, hex(-1 & mask))        # -1 thành 32 bit toàn 1
x = -6 & mask
print(x, x > 0x7FFFFFFF)                # bit dấu bật
print(~(x ^ mask))                      # đổi lại thành -6

4294967295 0xffffffff
4294967290 True
-6


### 3.7. Tính tay bù hai của `-6` (dùng 8 bit cho gọn)

Quy tắc: **bù hai của `-x`** = đảo mọi bit của `x` rồi cộng `1`.

- `6 = 00000110`
- đảo bit: `11111001`
- cộng `1`: `11111010` — đây là biểu diễn của `-6` trong 8 bit (`250` nếu đọc như số không dấu, vì `256 - 6 = 250`).

Nhìn theo cách khác: `-6` trong `n` bit chính là `2**n - 6`. Đó là lý do phép cộng bit "vòng quanh" modulo `2**n` vẫn cho kết quả đúng cho cả số âm: `-6 + 4` trong 8 bit là `250 + 4 = 254 = 11111110`, tức `256 - 2`, chính là `-2`.

In [8]:
x = -6
print(format(x & 0xFF, "08b"), x & 0xFF, 2**8 - 6)
y = (x & 0xFF) + 4
print(format(y, "08b"), y, "-> đọc như số có dấu:", y - 256)

11111010 250 250
11111110 254 -> đọc như số có dấu: -2


## 4. Ý tưởng nền tảng — tính bằng tay

Với `num1 = 12`, `num2 = 5`: đơn giản là `12 + 5 = 17`.

Với `num1 = -10`, `num2 = 4`: ta có thể nghĩ "đi từ `-10` lên `4` bước": `-9, -8, -7, -6`. Vậy kết quả `-6`.

Hai cách nhìn:

1. **Công thức:** dùng toán tử `+` (CPU có sẵn mạch cộng).
2. **Mô phỏng:** `num1` cộng `num2` nghĩa là bước `|num2|` lần, mỗi lần `+1` (nếu `num2 > 0`) hoặc `-1` (nếu `num2 < 0`).

Cách 2 chậm nhưng cho ta thấy phép cộng chỉ là "đếm tiếp".

In [9]:
a, b = -10, 4
cur = a
for step in range(b):
    cur += 1
    print(f"bước {step + 1}: {cur}")
print("kết quả:", cur)

bước 1: -9
bước 2: -8
bước 3: -7
bước 4: -6
kết quả: -6


## 5. Cách 1 — Brute Force: tăng/giảm từng đơn vị

Mô phỏng đúng ý nghĩa "đếm tiếp": nếu `num2 > 0` thì cộng `1` vào `num1` đúng `num2` lần; nếu `num2 < 0` thì trừ `1` đúng `|num2|` lần.

In [10]:
class SolutionLoop:
    def sum(self, num1, num2):
        step = 1 if num2 > 0 else -1
        for _ in range(abs(num2)):
            num1 += step
        return num1

sol = SolutionLoop()
print(sol.sum(12, 5))
print(sol.sum(-10, 4))
print(sol.sum(7, -9))

17
-6
-2


**Độ phức tạp:** Time `O(|num2|)`, Space `O(1)`. Với `|num2| <= 100` thì nhanh, nhưng nếu `num2` lớn tới `10^9` thì rất chậm.

**Nút thắt:** ta làm `|num2|` bước nhỏ trong khi chỉ cần **một** phép cộng. Phần cứng đã có sẵn mạch cộng, hoặc ta có thể mô phỏng mạch cộng bằng bit trong `O(số bit)`.

## 6. Ý tưởng thuật toán — bước đột phá

**Insight 1 (thực tế):** Python có toán tử `+` cho `int`. Đây là lời giải nên dùng — đơn giản, đúng, `O(1)` cho số nhỏ.

**Insight 2 (học thuật):** có thể cộng **không cần** dấu `+` bằng cách lặp:

- `s = a ^ b` (tổng bỏ nhớ), `c = (a & b) << 1` (nhớ).
- Bây giờ `a + b = s + c`. Gán `a, b = s, c` rồi lặp lại.
- Dừng khi **không còn nhớ** (`b == 0`), lúc đó `a` là tổng.

Mỗi vòng, nhớ bị đẩy sang trái ít nhất một bit nên sau tối đa `~32` vòng thì hết nhớ.

Quy tắc "nếu… thì…": **nếu** `b != 0` **thì** còn nhớ cần cộng tiếp; **nếu** `b == 0` **thì** xong.

**Invariant:** ở mọi vòng, `a + b` luôn bằng tổng ban đầu.

## 7. Các cách tiếp theo

### Cách 2 — Toán tử `+` (lời giải chuẩn)

Một dòng. Python lo hết: số âm, số lớn, mọi trường hợp.

In [11]:
class SolutionPlus:
    def sum(self, num1, num2):
        return num1 + num2

sol = SolutionPlus()
print(sol.sum(12, 5))
print(sol.sum(-10, 4))
print(sol.sum(-100, 100))

17
-6
0


**Độ phức tạp:** Time `O(1)` (số nhỏ), Space `O(1)`.

### Cách 3 — Cộng bằng thao tác bit (XOR + carry), xử lý số âm bằng mặt nạ 32 bit

Lặp: `a, b = (a ^ b) & mask, ((a & b) << 1) & mask` tới khi `b == 0`. Sau cùng nếu `a` có bit dấu (`a > 0x7FFFFFFF`) thì đổi về số âm.

In [12]:
class SolutionBits:
    def sum(self, num1, num2):
        mask = 0xFFFFFFFF
        a, b = num1 & mask, num2 & mask
        while b:
            a, b = (a ^ b) & mask, ((a & b) << 1) & mask
        return a if a <= 0x7FFFFFFF else ~(a ^ mask)

sol = SolutionBits()
print(sol.sum(12, 5))
print(sol.sum(-10, 4))
print(sol.sum(-7, -8))

17
-6
-15


**Độ phức tạp:** Time `O(B)` với `B = 32` là số bit (mỗi vòng đẩy nhớ lên ít nhất một bit), Space `O(1)`. Đây là nền tảng của bài LeetCode 371 "Sum of Two Integers".

### Cách 4 — Dùng `sum([...])` hoặc `operator.add`

Hàm có sẵn `sum` cộng mọi phần tử của một iterable; `operator.add(a, b)` là dạng hàm của `a + b`. Hữu ích khi cần truyền "phép cộng" như một hàm (ví dụ vào `reduce`).

In [13]:
from operator import add
from functools import reduce

class SolutionBuiltin:
    def sum(self, num1, num2):
        return sum([num1, num2])

print(SolutionBuiltin().sum(12, 5))
print(add(-10, 4))
print(reduce(add, [12, 5, -3]))

17
-6
14


**Độ phức tạp:** Time `O(1)`, Space `O(1)` (một list nhỏ hai phần tử).

## 8. TRACE từng bước

Trace cách 3 (bit) với `num1 = 5`, `num2 = 3` (dùng 4 bit để dễ nhìn):

| Vòng | a | b | a ^ b | (a & b) << 1 | a mới | b mới |
|---|---|---|---|---|---|---|
| 1 | `0101` | `0011` | `0110` | `0010` | `0110` | `0010` |
| 2 | `0110` | `0010` | `0100` | `0100` | `0100` | `0100` |
| 3 | `0100` | `0100` | `0000` | `1000` | `0000` | `1000` |
| 4 | `0000` | `1000` | `1000` | `0000` | `1000` | `0000` |

`b = 0` ⇒ dừng, `a = 1000 = 8`. Hàm dưới in từng vòng (cho cả số âm).

In [14]:
def trace_bits(num1, num2):
    mask = 0xFFFFFFFF
    a, b = num1 & mask, num2 & mask
    r = 0
    while b:
        r += 1
        s, c = (a ^ b) & mask, ((a & b) << 1) & mask
        print(f"vòng {r}: a={a:032b}\n         b={b:032b}\n         xor={s:032b} carry={c:032b}")
        a, b = s, c
    res = a if a <= 0x7FFFFFFF else ~(a ^ mask)
    print("kết quả =", res)
    return res

trace_bits(5, 3)

vòng 1: a=00000000000000000000000000000101
         b=00000000000000000000000000000011
         xor=00000000000000000000000000000110 carry=00000000000000000000000000000010
vòng 2: a=00000000000000000000000000000110
         b=00000000000000000000000000000010
         xor=00000000000000000000000000000100 carry=00000000000000000000000000000100
vòng 3: a=00000000000000000000000000000100
         b=00000000000000000000000000000100
         xor=00000000000000000000000000000000 carry=00000000000000000000000000001000
vòng 4: a=00000000000000000000000000000000
         b=00000000000000000000000000001000
         xor=00000000000000000000000000001000 carry=00000000000000000000000000000000
kết quả = 8


8

In [15]:
def trace_bits_short(num1, num2):
    mask = 0xFFFFFFFF
    a, b = num1 & mask, num2 & mask
    r = 0
    while b:
        r += 1
        a, b = (a ^ b) & mask, ((a & b) << 1) & mask
        print(f"vòng {r}: a={a} b={b}")
    res = a if a <= 0x7FFFFFFF else ~(a ^ mask)
    print("kết quả =", res)
    return res

trace_bits_short(-10, 4)

vòng 1: a=4294967282 b=8
vòng 2: a=4294967290 b=0
kết quả = -6


-6

## 9. Code nộp LeetCode

Dùng cách `+`. Chú ý tên phương thức đúng là `sum`.

In [16]:
from typing import List

class Solution:
    def sum(self, num1: int, num2: int) -> int:
        return num1 + num2

print(Solution().sum(12, 5))
print(Solution().sum(-10, 4))

17
-6


## 10. Vì sao thuật toán đúng?

- **Cách `+`:** là định nghĩa của phép cộng số nguyên, Python đảm bảo đúng.
- **Cách vòng lặp:** mỗi bước đổi tổng đúng `±1`, sau `|num2|` bước ta đã cộng chính xác `num2`.
- **Cách bit:** với mỗi cột, `a ^ b` cho bit tổng và `(a & b) << 1` cho bit nhớ chuyển sang cột trái. Như vậy `a + b = (a ^ b) + ((a & b) << 1)` — đây là đẳng thức đúng của bộ cộng nhị phân. Vì vế phải là tổng của hai số mới có "nhớ" nằm ở các bit cao hơn, lặp lại tới khi nhớ bằng `0` thì phần còn lại chính là tổng. Mỗi vòng, bit nhớ thấp nhất dịch lên ít nhất một vị trí, nên sau tối đa `32` vòng thì hết nhớ (với mặt nạ 32 bit).
- **Số âm:** mặt nạ `0xFFFFFFFF` biểu diễn số âm theo bù hai 32 bit; phép cộng bit trên số bù hai cho kết quả bù hai đúng (modulo `2^32`). Khi `a > 0x7FFFFFFF` thì bit dấu bật, `~(a ^ mask)` khôi phục đúng giá trị âm.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vòng tăng/giảm | `O(|num2|)` | `O(1)` |
| 2. Toán tử `+` | `O(1)` | `O(1)` |
| 3. Bit (XOR + carry) | `O(B)`, `B = 32` | `O(1)` |
| 4. `sum([...])` / `operator.add` | `O(1)` | `O(1)` |

Với constraints `-100..100`, mọi cách đều tức thời. Cách `+` là lựa chọn thực tế; cách bit giúp hiểu cấu tạo máy tính.

## 12. Bộ test

Test ví dụ đề, edge case; sau đó **kiểm tra toàn bộ** mọi cặp trong miền constraints (vì miền nhỏ nên vét cạn được), rồi random mở rộng với số lớn.

In [17]:
solutions = [SolutionLoop(), SolutionPlus(), SolutionBits(), SolutionBuiltin(), Solution()]

for s in solutions:
    assert s.sum(12, 5) == 17
    assert s.sum(-10, 4) == -6
print("ví dụ đề: OK")

ví dụ đề: OK


In [18]:
for s in solutions:
    assert s.sum(0, 0) == 0
    assert s.sum(0, 7) == 7
    assert s.sum(7, 0) == 7
    assert s.sum(-100, -100) == -200
    assert s.sum(100, 100) == 200
    assert s.sum(-100, 100) == 0
    assert s.sum(1, -1) == 0
    assert s.sum(-1, -1) == -2
print("edge case: OK")

edge case: OK


In [19]:
count = 0
for a in range(-100, 101):
    for b in range(-100, 101):
        expected = a + b
        for s in solutions:
            assert s.sum(a, b) == expected, (type(s).__name__, a, b)
        count += 1
print("vét cạn", count, "cặp trong [-100, 100]: tất cả khớp")

vét cạn 40401 cặp trong [-100, 100]: tất cả khớp


In [20]:
import random
random.seed(2235)
bits = SolutionBits()
for _ in range(2000):
    a, b = random.randint(-10**9, 10**9), random.randint(-10**9, 10**9)
    # cách bit chỉ đúng khi kết quả vừa trong 32 bit có dấu, nên ta chọn miền an toàn: |a + b| < 2**31
    assert bits.sum(a, b) == a + b, (a, b)
print("random số lớn (cách bit) khớp phép +: OK")

random số lớn (cách bit) khớp phép +: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên `self` hoặc gọi sai cách

Phương thức trong lớp cần `self` ở tham số đầu tiên; thiếu `self` làm `num1` bị coi là `self`, dẫn tới `TypeError` khi gọi.

In [21]:
class Wrong:
    def sum(num1, num2):          # thiếu self
        return num1 + num2

try:
    print(Wrong().sum(1, 2))
except TypeError as e:
    print("TypeError:", e)
print("đúng:", Solution().sum(1, 2))

TypeError: Wrong.sum() takes 2 positional arguments but 3 were given
đúng: 3


### Lỗi 2 — Dùng `print` thay vì `return`

LeetCode lấy **giá trị trả về**, không đọc `print`. Nếu quên `return`, hàm trả `None`.

In [22]:
class WrongPrint:
    def sum(self, num1, num2):
        print(num1 + num2)        # in ra nhưng không trả về

result = WrongPrint().sum(12, 5)
print("giá trị trả về:", result)  # None
print("đúng:", Solution().sum(12, 5))

17
giá trị trả về: None
đúng: 17


### Lỗi 3 — Vòng lặp vô hạn khi cộng bit với số âm mà không dùng mặt nạ

Python không giới hạn số bit, nên với số âm, nhớ có thể dịch trái mãi. Ta mô phỏng bằng giới hạn số vòng để không treo máy.

In [23]:
def wrong_bits_no_mask(a, b, limit=70):
    steps = 0
    while b and steps < limit:        # limit chỉ để demo, không treo
        a, b = a ^ b, (a & b) << 1
        steps += 1
    return a, b, steps

a, b, steps = wrong_bits_no_mask(-1, 1)
print("không mặt nạ: sau", steps, "vòng, b vẫn =", b != 0)
print("đúng:", SolutionBits().sum(-1, 1))

không mặt nạ: sau 70 vòng, b vẫn = True
đúng: 0


### Lỗi 4 — Quên đổi lại số âm sau khi dùng mặt nạ

Mặt nạ 32 bit cho ra số dương rất lớn (ví dụ `4294967290`); phải đổi lại bằng `~(a ^ mask)` khi `a > 0x7FFFFFFF`.

In [24]:
def wrong_no_convert(num1, num2):
    mask = 0xFFFFFFFF
    a, b = num1 & mask, num2 & mask
    while b:
        a, b = (a ^ b) & mask, ((a & b) << 1) & mask
    return a                          # SAI: thiếu bước đổi dấu

print("sai :", wrong_no_convert(-10, 4))        # 4294967290
print("đúng:", SolutionBits().sum(-10, 4))      # -6

sai : 4294967290
đúng: -6


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Bài kiểu "tính toán trực tiếp" ⇒ dùng toán tử có sẵn, đừng làm phức tạp.
- Đọc kỹ **tên phương thức và tham số** của LeetCode; nộp đúng `class Solution`, đúng chữ ký.
- Gặp "không dùng `+`/`-`" ⇒ nhớ cặp **XOR (tổng không nhớ) và AND << 1 (nhớ)**.
- Với Python, số âm + thao tác bit ⇒ nhớ **mặt nạ `0xFFFFFFFF`** và đổi dấu ở cuối.
- Trong phỏng vấn, hãy nói rõ: "Với Python tôi dùng `+`; nếu đề cấm `+` tôi mô phỏng bộ cộng bằng XOR và AND" — vừa ngắn gọn vừa cho thấy bạn hiểu bản chất.
- Luôn tự gõ một ví dụ có **số âm** và một ví dụ có **số 0** để kiểm tra trước khi nộp, dù bài trông đơn giản.
- Họ hàng: Sum of Two Integers (371), Add Strings (415), Add Binary (67), Plus One (66), Add to Array-Form of Integer (989).

## 15. Biến thể của bài

1. **Không dùng `+`/`-` (LC 371):** chính là cách 3, với phạm vi `-1000..1000`.
2. **Cộng hai chuỗi số lớn (LC 415):** cộng từng chữ số, nhớ sang cột trái.
3. **Cộng hai số nhị phân dạng chuỗi (LC 67).**
4. **Cộng một số vào mảng chữ số (LC 66, 989).**
5. **Tổng nhiều số:** `sum(list)`; hoặc `reduce(add, list)`.

In [25]:
# Biến thể 2: cộng hai chuỗi số (Add Strings) bằng cộng từng cột
def add_strings(x, y):
    i, j, carry, out = len(x) - 1, len(y) - 1, 0, []
    while i >= 0 or j >= 0 or carry:
        s = carry
        if i >= 0:
            s += int(x[i]); i -= 1
        if j >= 0:
            s += int(y[j]); j -= 1
        out.append(str(s % 10))
        carry = s // 10
    return "".join(reversed(out))

print(add_strings("999", "1"), add_strings("123", "456"))

1000 579


### Mở rộng: bộ cộng lan truyền (ripple-carry) trên danh sách bit

Để thấy rõ cơ chế mạch cộng, ta có thể biểu diễn mỗi số là danh sách bit (bit thấp trước) rồi cộng từng cột bằng "bộ cộng đầy đủ" (full adder): tổng bit = `a ^ b ^ carry`, nhớ mới = `(a & b) | (carry & (a ^ b))`.

In [26]:
def to_bits(x, width=8):
    return [(x >> i) & 1 for i in range(width)]          # bit thấp trước

def from_bits(bits):
    return sum(b << i for i, b in enumerate(bits))

def ripple_add(a_bits, b_bits):
    out, carry = [], 0
    for a, b in zip(a_bits, b_bits):
        out.append(a ^ b ^ carry)                         # bit tổng
        carry = (a & b) | (carry & (a ^ b))               # bit nhớ mới
    return out

print(to_bits(13), to_bits(7))
print(ripple_add(to_bits(13), to_bits(7)), from_bits(ripple_add(to_bits(13), to_bits(7))))
for x in range(0, 100, 7):
    for y in range(0, 100, 11):
        assert from_bits(ripple_add(to_bits(x), to_bits(y))) == x + y
print("ripple-carry khớp phép + trên các cặp thử")

[1, 0, 1, 1, 0, 0, 0, 0] [1, 1, 1, 0, 0, 0, 0, 0]
[0, 0, 1, 0, 1, 0, 0, 0] 20
ripple-carry khớp phép + trên các cặp thử


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vòng tăng/giảm | Đếm tiếp `|num2|` lần | `O(|num2|)` | `O(1)` | Hiểu bản chất phép cộng |
| 2. Toán tử `+` | Phép cộng có sẵn | `O(1)` | `O(1)` | **Nộp bài**, thực tế |
| 3. Bit XOR + carry | Mô phỏng mạch cộng | `O(B)` | `O(1)` | Đề cấm `+`, phỏng vấn |
| 4. `sum([...])`/`add` | Hàm có sẵn | `O(1)` | `O(1)` | Truyền hàm cộng vào `reduce`/`map` |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `Solution().sum(-3, -4)` bằng bao nhiêu?

**Bài 2 (tính tay).** Cộng nhị phân `1101` và `0111` bằng tay, đổi kết quả sang thập phân.

**Bài 3 (điền TODO).** Hoàn thành `half_adder(a, b)` trả về cặp `(tổng_không_nhớ, nhớ)` cho **hai bit** `a`, `b` (mỗi bit là `0` hoặc `1`).

**Bài 4 (tự viết lại).** Viết `add_no_plus(a, b)` cho **số không âm** bằng vòng `while b:` với XOR/AND (không cần mặt nạ).

**Bài 5 (mini challenge).** Viết `sum_many(nums)` cộng một danh sách bằng `reduce(add, ...)` và trả `0` nếu danh sách rỗng.

In [27]:
# Bài 1: đoán rồi chạy
print(Solution().sum(-3, -4))

-7


In [28]:
# Bài 3, 4, 5: TODO - KHÔNG được gọi khi run all
def half_adder(a, b):
    # TODO: trả về (a ^ b, a & b)
    pass

def add_no_plus(a, b):
    # TODO: cộng hai số không âm bằng XOR và AND << 1
    pass

def sum_many(nums):
    # TODO: dùng functools.reduce và operator.add, rỗng -> 0
    pass

## 18. Gợi ý 3 mức

**Bài 3**
- Mức 1: XOR cho bit tổng khi không tính nhớ.
- Mức 2: AND cho bit nhớ (cả hai bit đều `1`).
- Mức 3: `return a ^ b, a & b`.

**Bài 4**
- Mức 1: Lặp tới khi không còn nhớ (`b == 0`).
- Mức 2: Mỗi vòng `a, b = a ^ b, (a & b) << 1`.
- Mức 3: `while b: a, b = a ^ b, (a & b) << 1` rồi `return a`.

**Bài 5**
- Mức 1: `reduce(hàm, iterable, giá_trị_đầu)` có tham số thứ ba làm giá trị khởi tạo.
- Mức 2: Dùng `0` làm giá trị khởi tạo để danh sách rỗng trả `0`.
- Mức 3: `return reduce(add, nums, 0)`.

## 19. Đáp án tham khảo cho bài tập

Mỗi đáp án được kiểm tra bằng `assert`.

In [29]:
# Bài 1
assert Solution().sum(-3, -4) == -7

# Bài 2: 1101 (13) + 0111 (7) = 10100 (20)
assert 0b1101 + 0b0111 == 0b10100 == 20

# Bài 3
def half_adder_ans(a, b):
    return a ^ b, a & b
assert half_adder_ans(0, 0) == (0, 0)
assert half_adder_ans(1, 0) == (1, 0)
assert half_adder_ans(1, 1) == (0, 1)

# Bài 4
def add_no_plus_ans(a, b):
    while b:
        a, b = a ^ b, (a & b) << 1
    return a
assert add_no_plus_ans(5, 3) == 8
assert add_no_plus_ans(0, 9) == 9
assert add_no_plus_ans(13, 7) == 20

# Bài 5
def sum_many_ans(nums):
    return reduce(add, nums, 0)
assert sum_many_ans([12, 5, -3]) == 14
assert sum_many_ans([]) == 0
print("Tất cả đáp án bài tập: OK")

Tất cả đáp án bài tập: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

### Template

```python
class Solution:
    def sum(self, num1: int, num2: int) -> int:
        return num1 + num2

# Phiên bản bit (khi đề cấm dấu +):
#   mask = 0xFFFFFFFF
#   a, b = num1 & mask, num2 & mask
#   while b: a, b = (a ^ b) & mask, ((a & b) << 1) & mask
#   return a if a <= 0x7FFFFFFF else ~(a ^ mask)
```

### Checklist

- [ ] Tên phương thức đúng `sum`, có `self`, trả về `int`.
- [ ] Dùng `return`, không chỉ `print`.
- [ ] Hiểu `XOR` = tổng không nhớ, `AND << 1` = nhớ.
- [ ] Với số âm và thao tác bit: dùng mặt nạ `0xFFFFFFFF` và đổi dấu ở cuối.
- [ ] Đã test `0`, số âm, `-100`, `100`.
- [ ] Biết giới hạn của cách vòng lặp (`O(|num2|)`).